# Assignment 4

## Part 1: The dataset

### ⚙  Task 1.1. Downloading and inspecting the question answering dataset

In [ ]:
!curl -O https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2524k  100 2524k    0     0  6634k      0 --:--:-- --:--:-- --:--:-- 6625k


In [14]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [15]:
questions.iloc[0].question

'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?'

In [16]:
documents.iloc[0].abstract

'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos a

## Step 2: Configure your LangChain LM

### ⚙  Task 2.1. Select a language model

In [19]:
from langchain_community.llms import HuggingFacePipeline

model = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "return_full_text": False,
    },
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5035.93it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [23]:
#sanity check
prompt = "<|system|>\nYou are a helpful assistant.</s>\n<|user|>\nWhat causes type 2 diabetes?</s>\n<|assistant|>\n"
response = model.invoke(prompt)
print(response)

Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Type 2 diabetes is caused by a combination of genetic factors, lifestyle factors (such as unhealthy diet, lack of exercise, smoking, sedentary lifestyle, and high blood pressure), and environmental factors (such as infection, hormonal changes, and stress). Here are some of the main lifestyle factors that contribute to type 2 diabetes:

1. Inactivity: People with type 2 diabetes who are physically inactive, such as not exercising regularly, may have a higher risk of developing the condition.

2. Obesity: Overweight or obese individuals are at higher risk of developing type 2 diabetes.

3. High blood pressure: High blood pressure, or hypertension, is a risk factor for type 2 diabetes. Chronic high blood pressure can lead to a build-up of fatty deposits in the walls of blood vessels, which can increase the risk of developing type 2 diabetes.

4. Low-level insulin: In people with type 2 diabetes, the body produces less insulin, which is a hormone that


COmment: I chose tiny llamas as it is small enough to try to run locally and instruction-tuned for chat/q&A

## Part 3: Set up the document database

### 🎓  Task 3.1. Embedding model (exam)

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5677.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
#sanity check 
# Sanity check
test_embedding = embedding_model.embed_query("What causes type 2 diabetes?")
print(np.array(test_embedding).shape)  # should be (384,)

(384,)


COmment: Using hugging face embeddings which loads a sentence transformer model that converts text into fixed-size vector so embed_query takes a string and and return a list of floats (the embedding). we choose all-minilm-l6-v2 that produces 384 dimensional vector as we see in sanity check. I choose this one as it's small and fast and compact enough (384 dim) to store meaningful semantics.

The reason for the embedding model is we can't do meaningful similarity search on raw text and we need to map both document and queries into shared vecyor space so semantic similarity is now geometric closeness. so embedding encodes meaning of text into dense vectors, like in this case what causes diabetes ends up close to insulin resistance w/o sharing words. 

### ⚙  Task 3.2. Chunking

Do chunking as some docs might be too long

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)
chunks = text_splitter.split_documents(texts)

In [33]:
# Sanity check
for chunk in chunks[:3]:
    print(chunk.page_content)
    print(f"chars: {len(chunk.page_content)}, id: {chunk.metadata['id']}")
    print("---")
print(f"Total chunks: {len(chunks)}")

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has
chars: 498, id: 21645374
---
has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window s

### 🎓  Task 3.3. Define a vector store (exam)

In [34]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

In [36]:
#sanity check
results = vector_store.similarity_search_with_score(
    "What causes type 2 diabetes?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.345430] Type 2 diabetes may be present for several years before diagnosis, by which time many patients have already developed diabetic complications. Earlier detection and treatment may reduce this burden, but evidence to support this approach is lacking. Glycemic control and clinical and surrogate outcomes were compared for 5,088 of 5,102 U.K. Diabetes Prospective Study participants according to whether they had low (<140 mg/dl [<7.8 mmol/l]), intermediate (140 to<180 mg/dl [7.8 to<10.0 mmol/l]), or [{'id': 12145243}]
* [SIM=0.409019] insulin resistance, which raises the question of whether glucose lowering per se without changes in the processes that underlie hyperglycemia should be the sole clinical paradigm in the treatment of type 2 diabetes or its prevention. [{'id': 22720085}]
* [SIM=0.425476] People presenting with type 2 diabetes with lower initial glycemia who may be earlier in the course of their disease had fewer adverse clinical outcomes despite similar glycemic p

Comment: chroma from doc embeds all our chunks using th embedding model and stores both text and its vector in memory chroma database. collection set as cosine similarity is the distance we use to retrieve by directinal similarity. in sanity check we want the three returned chunks to be about insulin, glucose etc. which we can see we get so it seems to work. All scores normalized between 0-1.

Other option beside cosine could be l2 which i euclidean distance, this is sensitive to vector magnitude and for text we care more about direction so what they mean rather than size.

## Part 4: Implementing the system

### 🎓  Task 4.1. Defining the full RAG pipeline

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

retriever = vector_store.as_retriever(search_kwargs={"k": 1}) #retriever is a runnable that takes a query and returns a list of documents, we set k=1 to get the top 1 chunk as context for the prompt, we could set k>1 to get more chunks but we want to keep it simple for now

def format_docs(docs): #takes list of documents and joins them into one string with two newlines in between, this is the format we want to pass to the prompt as context
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    "You are a biomedical research assistant. Use only the context below to answer.\n\n" #provide prompt with some context
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer based strictly on the context provided."
)

runnable_parallel_object = RunnableParallel( #runs two thins simultaneously, one is retriever (fetch top1 chunk) and other is format docs that takes list of documents from retrieved and joins into one string, then passes the output to prompt
    context=retriever | format_docs,
    question=RunnablePassthrough(), #passes q as is without any change to the prompt
)

chain = (
    prompt
    | model
    | StrOutputParser() #takes the output from the model and parses it as a string, this is the final answer we want to get
)

rag_chain = runnable_parallel_object.assign(answer=chain)

In [54]:
#look at raw data and get example q
import json
with open("ori_pqal.json") as f:
    data = json.load(f)

first_key = list(data.keys())[0]
print(data[first_key].keys())

dict_keys(['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER'])


In [53]:
#sanity check
my_query = data[first_key]["QUESTION"] #get out our q.
print(my_query)

result = rag_chain.invoke(my_query)
print("Question:", my_query)
print("\nContext:", result["context"])
print("\nAnswer:", result["answer"])

Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?


Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has

Answer: 

Reasoning:

PCD is regulated by mitochondria. Mitochondria break down the ATP content of the cell. During PCD, the plant produces ATP through the breakdown of glycolysis. The ATP produced through glycolysis is stored within the mitochondria. When PCD occurs, the mitochondria stop producing ATP, causing the plant to die. This would suggest that mitochondria play a role in remod

Comment: We use option B building the RAg on lang chain open tutorial as i was straightforward to apply than the agent middleware approach and easy to debug as each step clear

We use RAg to ensure our model is more domain specific so we ground the response so given a q we search pudmedqa abstract for relevant corpus using the embeddign model and our vector storage and add the passage into prompt context before asking our model to answer. 

What we do is to add a chat template asa biomedical assistance and ask model to fetch relevant q and answers. Results for mitochondria lace plan leaves is the right context is gathered and reasoning also connected to q and context. Interesting, we prompt to answer only from context so at the end we see "context does not specific whether mitochondria are involved in actual remodelling" is what we hope and expect to see. 

## Part 5: Evaluate RAG on the dataset

### 🎓  Task 5.1. High-level evaluation

In [55]:
import pandas as pd
import re
from sklearn.metrics import f1_score, accuracy_score
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#build questions dataframe (binary only so only yes/no)
questions = pd.DataFrame([
    {"question": v["QUESTION"], "gold_label": v["final_decision"]} #build a dataframe with two columns, question and gold label (yes/no/maybe)
    for v in data.values()
])
questions = questions[questions["gold_label"].isin(["yes", "no"])].reset_index(drop=True) #filter to only yes/no questions
print(f"Total binary questions: {len(questions)}")

def extract_yes_no(text):
    match = re.search(r'\b(yes|no)\b', text.lower()[:150])#search for the first occurrence of "yes" or "no" in the text, ignoring case and only looking at the first 150 characters to avoid long outputs that might contain irrelevant information
    return match.group(1) if match else None

#update RAG prompt to ask for yes/no classification
rag_prompt = ChatPromptTemplate.from_template(
    "You are a biomedical research assistant. Use only the context below.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer with only 'Yes' or 'No' based on the context."
)

chain = rag_prompt | model | StrOutputParser()
rag_chain = runnable_parallel_object.assign(answer=chain)

#baseline prompt (no context)
baseline_prompt = ChatPromptTemplate.from_template(
    "You are a biomedical research assistant.\n\n"
    "Question: {question}\n\n"
    "Answer with only 'Yes' or 'No'."
)
baseline_chain = baseline_prompt | model | StrOutputParser()

#run on a subset first to save time 
subset = questions.head(50)

rag_preds, baseline_preds, gold = [], [], []

for _, row in subset.iterrows():
    rag_result = rag_chain.invoke(row["question"])
    rag_pred = extract_yes_no(rag_result["answer"])

    baseline_result = baseline_chain.invoke({"question": row["question"]})
    baseline_pred = extract_yes_no(baseline_result)

    rag_preds.append(rag_pred)
    baseline_preds.append(baseline_pred)
    gold.append(row["gold_label"])

#filter to valid answers only
def evaluate(preds, labels):
    valid = [(p, l) for p, l in zip(preds, labels) if p is not None] #only keep the predictions that are valid (yes/no), if the model did not return a valid answer we ignore that prediction for evaluation, this is important because we want to compare the models based on their valid predictions and not penalize them for invalid outputs
    if not valid:
        return 0, 0, 0
    p, l = zip(*valid)
    return (
        len(valid),
        f1_score(l, p, pos_label="yes"),
        accuracy_score(l, p),
    )

rag_n, rag_f1, rag_acc = evaluate(rag_preds, gold) #evaluate RAG predictions against gold labels, this will give us the number of valid predictions, F1 score and accuracy for the RAG model
bas_n, bas_f1, bas_acc = evaluate(baseline_preds, gold)

print(f"\nRAG      — valid: {rag_n}/{len(subset)}, F1: {rag_f1:.3f}, Accuracy: {rag_acc:.3f}")
print(f"Baseline — valid: {bas_n}/{len(subset)}, F1: {bas_f1:.3f}, Accuracy: {bas_acc:.3f}")

Total binary questions: 890


Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


RAG      — valid: 22/50, F1: 0.765, Accuracy: 0.636
Baseline — valid: 20/50, F1: 0.812, Accuracy: 0.700


Comment: We change prompt and data to only have yes/no and specifically ask to give yes/no answers and compare to baseline with no context for first 50 samples from data. We compare to golden standard answer using F1 and accuracy wher F1 measures balance between precision and recall and accuracy factor of valid predicions that match gold label. 

Result we see that we have 22/50 valid for RAG and 20/50 for baseline so I would say it's comparable. Accuracy is higher for baseline and F1, which is not expected but we see that we can only judge less than 50% of the answers of my sample so it seems thats the real issue. I would except RA to be higher otherwise, but at the same time our TineLlama is so small so maybe RAG just does not add much it might just struggle with adding information from retrieved pagges. Also maybe for the questions, TineLlama might already know enough about biomed and simple yes/no it migh not need much else. 

### 🎓  Task 5.2. Detailed inspection

In [56]:
# Rebuild questions with gold_document_id
questions = pd.DataFrame([
    {"question": v["QUESTION"], "gold_label": v["final_decision"], "gold_document_id": k}  #build a dataframe with three columns, question, gold label (yes/no/maybe) and gold document id (the id of the document that contains the answer to the question, this is the document that we want to retrieve with our RAG model)
    for k, v in data.items()
])
questions = questions[questions["gold_label"].isin(["yes", "no"])].reset_index(drop=True) #filter to only yes/no questions
subset = questions.head(50)

# Evaluate retrieval hit rate
hits = []
inspection = []

for _, row in subset.iterrows(): #for each question in the subset, we run the retriever to get the top 1 document and check if it matches the gold document id, we also run the RAG chain to get the answer and extract yes/no from it, then we store the results for later analysis
    retrieved_docs = retriever.invoke(row["question"])
    retrieved_id = str(retrieved_docs[0].metadata["id"]) if retrieved_docs else None
    gold_id = str(row["gold_document_id"]) #the gold document id is the id of the document that contains the answer to the question, we compare it to the retrieved document id to see if we retrieved the correct document
    hit = retrieved_id == gold_id

    result = rag_chain.invoke(row["question"])
    pred = extract_yes_no(result["answer"])

    hits.append(hit)
    inspection.append({
        "question": row["question"],
        "gold_id": gold_id,
        "retrieved_id": retrieved_id,
        "hit": hit,
        "pred": pred,
        "gold_label": row["gold_label"],
        "context": retrieved_docs[0].page_content if retrieved_docs else "",
        "answer": result["answer"][:300],
    })

hit_rate = sum(hits) / len(hits)
print(f"Retrieval hit rate: {sum(hits)}/{len(hits)} ({hit_rate:.1%})\n")


Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Retrieval hit rate: 49/50 (98.0%)



In [61]:
#inspect a correct and incorrect retrieval
df_inspect = pd.DataFrame(inspection)

print("retrieval example 1")
ex = df_inspect[df_inspect["hit"] == True].iloc[0]
print(f"Q: {ex['question']}\nContext: {ex['context'][:300]}\nPred: {ex['pred']} | Gold: {ex['gold_label']}\n")

print("retrieval example 2")
ex = df_inspect[df_inspect["hit"] == True].iloc[5]
print(f"Q: {ex['question']}\nContext: {ex['context'][:300]}\nPred: {ex['pred']} | Gold: {ex['gold_label']}")

retrieval example 1
Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel
Pred: nan | Gold: yes

retrieval example 2
Q: Is adjustment for reporting heterogeneity necessary in sleep disorders?
Context: adjustment for differences in response category cut-points for each individual. The prevalence of self-reported problems with sleep and energy was 53 %. Without correction of cut-point shifts, age, sex, and the number of comorbidities were significantly associated with a greater severity of sleep-re
Pred: yes | Gold: no


Comment: We create a code to check how often retriever actually fecthes the right paper by ting dataframe with gold label and gold docs and seeing if it's fetched. If this is low, then I believe that chunking or embedding quality is the problem here. Retrieval hit rate is 98% so in this case it really does retireve the right documents and we can't blame anything else so pipeline seems correct for retrieving relevant docs but the model often does not follow yes/no instructions. Also to note that the pudmedqa question written from source abstract so quite easy to match in embedding space. 

From examples we see for mitochondria it gets the right doca nd answer with context and like previusly it cannot fully answr yes or no due to lack of explicit contextfrom abstract. For example 2 about heterogenity in sleep disorders it predicts yes and gives us an answer for it. 